In [ ]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
import pandas as pd
#import yfinance as yf
from duckduckgo_search import DDGS
from crewai.tools import BaseTool

In [2]:
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

In [3]:
llm = LLM(
    model="gemini-flash-latest",  #"gemini/gemini-2.0-flash",  # Ensure this model is valid and accessible
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.7
)

In [ ]:
class CSVReaderTool(BaseTool):
    name: str = "CSVReaderTool"
    description: str = "read and return a given number of records from a given feedback type"

    def _run(self, df_name: str, recno_start: int, recno_end: int):
        # Iterate over the DataFrame rows as (index, Series) pairs
        df = pd.read_csv(df_name)
        for index, row in df[recno_start:recno_end].iterrows():
        # Convert the row Series to a JSON string and yield it
        yield row.to_json()


In [ ]:
csv_reader_agent = Agent(
    role='Read CSV data',
    goal='Reads and parses feedback data from CSV files',
    backstory='Expert in reading a CSV file and returning the required records',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

feedback_classifier_agent = Agent(
    role='Classify feedback into one of the categories: bug, feature request, praise, complaint and spam data',
    goal='Categorize feedback into given categories',
    backstory='Expert in understanding customer issues from their feedback and categorizing those',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

bug_analysis_agent = Agent(
    role='Extract technical details: steps to reproduce, platform info, severity assessment - and output those in json format',
    goal='Extract technical details from feedback text provided it is classified as a bug',
    backstory='Expert in finding technical details from customer feedback',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

feature_extractor_agent = Agent(
    role='Identifies new feature requests and estimates user impact/demand from user feedback - and output those in json format',
    goal='Identify new feature requests from feedback text provided the feedback is classified as feature request',
    backstory='Expert in identifying feature requests from customer feedback',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

ticket_creactor_agent = Agent(
    role='Creates a list output containing the source_id, source_type, category, priority, technical_details and suggested_title using the outputs from other agents',
    goal='Create ticket details from feedback text',
    backstory='Expert in creating ticket details from customer feedback',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

quality_critic_agent = Agent(
    role='Creates a list output containing the source_id, source_type, category, priority, technical_details and suggested_title using the outputs from other agents',
    goal='Create ticket details from feedback text',
    backstory='Expert in creating ticket details from customer feedback',
    verbose=True,  # Keep agent verbose for debugging, we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)


In [ ]:
csv_reader_tool = CSVReaderTool()
#finance_tool = YahooFinanceTool()

csv_read_task = Task(
    description="read either the support_store_review or the support_feedback file, and return the requested records.",
    expected_output="JSON formatted records.",
    agent=csv_reader_agent,
    tools=[CSVReaderTool]
)

feedback_classifier_task = Task(
    description="identify the category of the feedback.",
    expected_output="A string containing one of - bug, feature request, praise, complaint, spam.",
    agent=feedback_classifier_agent
)

bug_analysis_task = Task(
    description="extracts technical details from a feedback, provided it is categorized as a bug by feedback_classifier_task.",
    expected_output="A json containing identified technical details.",
    agent=bug_analysis_agent
)

feature_extractor_task = Task(
    description="extracts features requested from a feedback, provided it is categorized as a feature request by feedback_classifier_task.",
    expected_output="A list containing identified features.",
    agent=feature_extractor_agent
)

ticket_creator_task = Task(
    description="Generates structured tickets and logs them to output CSV files.",
    expected_output="A list containing the ticket information.",
    agent=ticket_creactor_agent
)


In [ ]:
import pandas as pd
import json

def generate_json_rows(dataframe):
    """
    A generator function to yield one row at a time from a pandas DataFrame as a JSON formatted string.

    Args:  dataframe (pd.DataFrame): The input pandas DataFrame.

    Yields: str: A JSON formatted string representing a single row.
    """
    # Iterate over the DataFrame rows as (index, Series) pairs
    for index, row in dataframe.iterrows():
        # Convert the row Series to a JSON string and yield it
        yield row.to_json()

# --- Example Usage ---
# 1. Create a sample DataFrame
data = {
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'city': ['New York', 'Los Angeles', 'Chicago']
}
df = pd.DataFrame(data)

# 2. Use the generator function
print("Iterating through the DataFrame rows as JSON strings:")
for json_row in generate_json_rows(df[1:]):
    print(json_row)

Iterating through the DataFrame rows as JSON strings:
{"name":"Bob","age":30,"city":"Los Angeles"}
{"name":"Charlie","age":35,"city":"Chicago"}
